# Marathon Agent - Cloud Interactive Runner

Run your entire marathon AI coaching engine directly in the cloud from your phone, tablet, or laptop-**no home PC required**.

### Step 0: Add Your Garmin Credentials to Colab Secrets
1. Click the **Key icon (Secrets)** in the left sidebar of Google Colab.
2. Add a new secret with Name GARMIN_EMAIL and your Garmin email as the value.
3. Add a new secret with Name GARMIN_PASSWORD and your Garmin password as the value.
4. *(Optional)* Add GMAIL_APP_PASSWORD and GEMINI_API_KEY.
5. Toggle the switch to **grant notebook access** to each secret.


In [ ]:
#@title 1. Clone Repo & Install Dependencies
import os
import sys

!git clone https://github.com/maxxsotelo/marathon-agent.git 2>/dev/null || (cd marathon-agent && git pull)
%cd /content/marathon-agent
!pip install -r requirements.txt -q
print("Setup Complete: Environment is ready!")

In [ ]:
#@title 2. Load Credentials from Colab Secrets
from google.colab import userdata
import os

os.environ["GARMIN_EMAIL"] = userdata.get("GARMIN_EMAIL")
os.environ["GARMIN_PASSWORD"] = userdata.get("GARMIN_PASSWORD")

try:
    os.environ["GMAIL_APP_PASSWORD"] = userdata.get("GMAIL_APP_PASSWORD")
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

print(f"Authenticated as: {os.environ['GARMIN_EMAIL']}")

In [ ]:
#@title 3. Morning Telemetry Audit (HRV, Sleep, Readiness, ACWR)
!python sensor_fetch_garmin.py

In [ ]:
#@title 4. Run Pre-Schedule Tolerance Check (Rule 2 Independent Audit)
duration_minutes = 45 #@param {type:"integer"}
intensity = "easy" #@param ["recovery", "easy", "marathon", "threshold", "vo2max"]
target_date = "" #@param {type:"string"}

date_flag = f"--date {target_date}" if target_date else ""
!python sensor_pre_schedule_check.py --duration {duration_minutes} --intensity {intensity} {date_flag}

In [ ]:
#@title 5. Dispatch Daily Coaching Report to Email
!python actuator_send_report_email.py